# RNN Text Classification (LSTM vs GRU)

## الترتيب / Flow
1. استيراد المكتبات - Import libraries
2. قراءة البيانات - Load dataset
3. Tokenization + padding - Text to sequences
4. تقسيم البيانات - Train/Test split
5. بناء LSTM - Build LSTM model
6. تدريب LSTM - Train and evaluate
7. بناء GRU - Build GRU model
8. تدريب GRU - Train and evaluate
9. مقارنة النماذج - Compare accuracy

In [1]:
# Step 1) استيراد المكتبات / Import libraries
# pip install tensorflow -q  # uncomment in Colab if needed
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout

In [2]:
# Step 2) قراءة البيانات / Load dataset
dataset = pd.read_csv('sentiment_reviews.csv')
print(dataset['sentiment'].value_counts())
dataset.head()

sentiment
1    241
0    241
Name: count, dtype: int64


,review,sentiment
0,I loved this product it works perfectly,1
1,Terrible quality broke after one day,0
2,Amazing experience highly recommend,1
3,Worst purchase ever complete waste,0
4,Good value for money satisfied,1


In [3]:
# Step 3) Tokenization + padding / Text to sequences
MAX_WORDS = 2000
MAX_LEN = 20

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(dataset['review'])
sequences = tokenizer.texts_to_sequences(dataset['review'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = dataset['sentiment'].values
print('X shape:', X.shape)

X shape: (482, 20)


In [4]:
# Step 4) تقسيم البيانات / Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

In [5]:
# Step 5) بناء LSTM / Build LSTM model
def build_rnn_model(rnn_layer):
    model = Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
        rnn_layer,
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_rnn_model(LSTM(64))
lstm_model.summary()

C:\Users\A7MED\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Step 6) تدريب LSTM / Train LSTM
lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)
lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f'LSTM test accuracy: {lstm_acc:.2%}')

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.5032 - loss: 0.6941 - val_accuracy: 0.4935 - val_loss: 0.6940
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5195 - loss: 0.6926 - val_accuracy: 0.4935 - val_loss: 0.6934
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4870 - loss: 0.6948 - val_accuracy: 0.5065 - val_loss: 0.6927
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5195 - loss: 0.6931 - val_accuracy: 0.8052 - val_loss: 0.6920
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5747 - loss: 0.6887 - val_accuracy: 0.4935 - val_loss: 0.6862
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8506 - loss: 0.6033 - val_accuracy: 0.7922 - val_loss: 0.4950
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9935 - loss: 0.1466 - val_accuracy: 0.8961 - val_loss: 0.3616
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9903 - loss: 0.0489 - val_accuracy: 0.8701 - val_lo

In [7]:
# Step 7) بناء GRU / Build GRU model
gru_model = build_rnn_model(GRU(64))
gru_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Step 8) تدريب GRU / Train GRU
gru_history = gru_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)
gru_loss, gru_acc = gru_model.evaluate(X_test, y_test, verbose=0)
print(f'GRU test accuracy: {gru_acc:.2%}')

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.4708 - loss: 0.6941 - val_accuracy: 0.4935 - val_loss: 0.6933
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5162 - loss: 0.6927 - val_accuracy: 0.4935 - val_loss: 0.6933
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4708 - loss: 0.6933 - val_accuracy: 0.4935 - val_loss: 0.6933
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5260 - loss: 0.6917 - val_accuracy: 0.4935 - val_loss: 0.6934
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5292 - loss: 0.6942 - val_accuracy: 0.4935 - val_loss: 0.6932
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4870 - loss: 0.6943 - val_accuracy: 0.5065 - val_loss: 0.6931
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5195 - loss: 0.6927 - val_accuracy: 0.4935 - val_loss: 0.6931
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4675 - loss: 0.6930 - val_accuracy: 0.4935 - val_l

In [9]:
# Step 9) مقارنة + تنبؤ على جملة جديدة / Compare models + sample prediction
print('--- RNN Comparison ---')
print(f'LSTM: {lstm_acc:.2%}')
print(f'GRU:  {gru_acc:.2%}')

sample_reviews = [
    'I loved this product it works perfectly',
    'Terrible quality broke after one day'
]
sample_seq = pad_sequences(tokenizer.texts_to_sequences(sample_reviews), maxlen=MAX_LEN)
lstm_preds = (lstm_model.predict(sample_seq, verbose=0) > 0.5).astype(int).flatten()
gru_preds = (gru_model.predict(sample_seq, verbose=0) > 0.5).astype(int).flatten()

for i, text in enumerate(sample_reviews):
    print(f"\nReview: {text}")
    print(f"LSTM sentiment: {lstm_preds[i]} | GRU sentiment: {gru_preds[i]}")

--- RNN Comparison ---
LSTM: 65.98%
GRU:  49.48%

Review: I loved this product it works perfectly
LSTM sentiment: 1 | GRU sentiment: 1

Review: Terrible quality broke after one day
LSTM sentiment: 1 | GRU sentiment: 1
